In [ ]:
import math

import numpy as np
import pandas as pd

from scipy.stats import norm, ttest_ind
from statsmodels.stats.multitest import multipletests

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import xgboost as xgb


# Select the prepared dataset used for the analysis.
combined_dataset = pd.read_csv(r'dataset_1.csv')  


## Standardization

In [ ]:
features_to_scale = combined_dataset.drop(columns=['data', 'label'])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_to_scale)
scaled_df = pd.DataFrame(scaled_features, columns=features_to_scale.columns)
scaled_df['data'] = combined_dataset['data'].values
scaled_df['label'] = combined_dataset['label'].values
scaled_df = scaled_df[['data'] + ['label'] + list(features_to_scale.columns) ]
combined_dataset = scaled_df
cleaned_data = combined_dataset.drop(columns=['data'])


## Statistical feature selection

In [ ]:
group_0 = combined_dataset[combined_dataset['label'] == 0].drop(columns=['data', 'label'])
group_1 = combined_dataset[combined_dataset['label'] == 1].drop(columns=['data', 'label'])

t_stats, p_values = ttest_ind(group_0, group_1, equal_var=False)  

ttest_results = pd.DataFrame({
    'feature': group_0.columns,
    't_stat': t_stats,
    'p_value': p_values
})

p_values_df = pd.DataFrame({
    'feature': ttest_results['feature'],
    'p_value': ttest_results['p_value']
})

# Benjamini-Hochberg false discovery rate correction.
_, q_vals, _, _ = multipletests(pvals=p_values_df['p_value'], method='fdr_bh')
p_values_df["q_value"] = q_vals



In [ ]:
MANUAL_K_CLUSTERS = 10
FDR_ALPHA = 0.05
CHECKPOINTS = [2, 3, 4]       
MAX_ITER = max(CHECKPOINTS)


def run_iterations(cleaned_data, ttest_results, k_clusters, max_iter, checkpoints):
    """Run recursive clustering and return snapshots at checkpoint iterations."""
    current_data = cleaned_data.copy()
    original_feature_order = [col for col in cleaned_data.columns if col != 'label']
    snapshots = {}

    for i in range(max_iter):
        print(f"\n[{i+1}/{max_iter}] Iteration started...")

        non_feature_cols = [col for col in ['label'] if col in current_data.columns]
        feature_cols_df = current_data.drop(columns=non_feature_cols)
        X_features = feature_cols_df.T

        if len(X_features) < k_clusters:
            print(f"Warning: The remaining number of features ({len(X_features)}) is less than k_clusters ({k_clusters}). Stopping the loop.")
            break

        print(f"-> {len(X_features)} feature, partitioned into {k_clusters} clusters using KMeans...")
        kmeans = KMeans(n_clusters=k_clusters, n_init=10, random_state=42)
        cluster_labels = kmeans.fit_predict(X_features)

        feature_clusters = pd.DataFrame({
            "feature_name": X_features.index,
            "cluster": cluster_labels
        })

        groups_stat_list = []
        cluster_features_map = {}

        for c in range(k_clusters):
            cluster_features = feature_clusters[feature_clusters["cluster"] == c]["feature_name"].tolist()
            cluster_features_map[c] = cluster_features

            matching_rows = ttest_results[ttest_results['feature'].isin(cluster_features)]
            N_g = len(matching_rows)
            sum_g = matching_rows['t_stat'].sum() if N_g > 0 else 0

            z_g = sum_g / math.sqrt(N_g) if N_g > 0 else 0
            p_value_bidirectional = 2 * (1 - norm.cdf(abs(z_g)))

            groups_stat_list.append({
                'group': f"g{c+1}",
                'p_value': p_value_bidirectional,
                'cluster_index': c
            })

        groups_stat = pd.DataFrame(groups_stat_list)
        _, q_values, _, _ = multipletests(groups_stat['p_value'], method='fdr_bh')
        groups_stat['q_value'] = q_values

        significant_groups = groups_stat[groups_stat['q_value'] <= FDR_ALPHA]

        if len(significant_groups) == 0:   
            print("No significant cluster (q == 0) was found in this iteration. Stopping feature filtering.")
            break

        selected_features = []
        for c in significant_groups['cluster_index']:
            selected_features.extend(cluster_features_map[c])

        selected_feature_set = set(selected_features)
        ordered_selected_features = [col for col in original_feature_order if col in selected_feature_set]

        print(f"-> Number of significant clusters selected: {len(significant_groups)}")
        print(f"-> Total number of features in the new dataset: {len(ordered_selected_features)}")

        current_data = current_data[non_feature_cols + ordered_selected_features]

        if (i + 1) in checkpoints:
            snapshots[i + 1] = current_data.copy()

    # If the loop stops early, use the last available dataset for unreachable checkpoints
    for cp in checkpoints:
        if cp not in snapshots:
            print(f"Note: Iteration {cp} was not reached; the last available dataset is being used.")
            snapshots[cp] = current_data.copy()

    return snapshots


def evaluate_svc(new_cleaned_data):
    X = new_cleaned_data.drop(["data", "label"], axis=1, errors="ignore")
    y = new_cleaned_data["label"]

    model = SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        probability=True,
        shrinking=True,
        random_state=42
    )

    scoring = {
        'accuracy': 'accuracy',
        'f1': 'f1',
        'precision': 'precision',
        'recall': 'recall',
        'roc_auc': 'roc_auc',
        'mcc': make_scorer(matthews_corrcoef)
    }

    return cross_validate(
        model, X, y,
        cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )


def mean_std(scores):
    return f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"


# 1) Run the iterations and collect datasets at the checkpoints
snapshots = run_iterations(cleaned_data, ttest_results, MANUAL_K_CLUSTERS, MAX_ITER, CHECKPOINTS)

# 2) Evaluate and print the SVC results for each dataset
summary_rows = []

for n_iter in CHECKPOINTS:
    new_cleaned_data = snapshots[n_iter]
    n_features = new_cleaned_data.drop(["data", "label"], axis=1, errors="ignore").shape[1]

    cv_results = evaluate_svc(new_cleaned_data)

    print("\n" + "=" * 55)
    print(f"Number of iterations: {n_iter} | Number of features: {n_features} | Dataset shape: {new_cleaned_data.shape}")
    print("=" * 55)
    print("10-Fold CV Results (SVC):")
    print(f"Accuracy     : {mean_std(cv_results['test_accuracy'])}")
    print(f"F1 Score     : {mean_std(cv_results['test_f1'])}")
    print(f"Sensitivity  : {mean_std(cv_results['test_recall'])}")
    print(f"Precision    : {mean_std(cv_results['test_precision'])}")
    print(f"AUC (ROC)    : {mean_std(cv_results['test_roc_auc'])}")
    print(f"MCC          : {mean_std(cv_results['test_mcc'])}")

    summary_rows.append({
        "iterations": n_iter,
        "n_features": n_features,
        "Accuracy": mean_std(cv_results['test_accuracy']),
        "F1": mean_std(cv_results['test_f1']),
        "Sensitivity": mean_std(cv_results['test_recall']),
        "Precision": mean_std(cv_results['test_precision']),
        "AUC": mean_std(cv_results['test_roc_auc']),
        "MCC": mean_std(cv_results['test_mcc']),
    })

# 3) Comparison table
print("\n\n--- SUMMARY TABLE ---")
summary_df = pd.DataFrame(summary_rows).set_index("iterations")
print(summary_df.to_string())

In [ ]:
MANUAL_K_CLUSTERS = 10
FDR_ALPHA = 0.05
CHECKPOINTS = [2, 3, 4]          # checkpoints to report
MAX_ITER = max(CHECKPOINTS)


def run_iterations(cleaned_data, ttest_results, k_clusters, max_iter, checkpoints):
    """Run recursive clustering and return snapshots at checkpoint iterations."""
    current_data = cleaned_data.copy()
    original_feature_order = [col for col in cleaned_data.columns if col != 'label']
    snapshots = {}

    for i in range(max_iter):
        print(f"\n[{i+1}/{max_iter}] Iteration started...")

        non_feature_cols = [col for col in ['label'] if col in current_data.columns]
        feature_cols_df = current_data.drop(columns=non_feature_cols)
        X_features = feature_cols_df.T

        if len(X_features) < k_clusters:
            print(f"Warning: The remaining number of features ({len(X_features)}) is less than k_clusters ({k_clusters}). Stopping the loop.")
            break

        print(f"-> {len(X_features)} feature, partitioned into {k_clusters} clusters using KMeans...")
        kmeans = KMeans(n_clusters=k_clusters, n_init=10, random_state=42)
        cluster_labels = kmeans.fit_predict(X_features)

        feature_clusters = pd.DataFrame({
            "feature_name": X_features.index,
            "cluster": cluster_labels
        })

        groups_stat_list = []
        cluster_features_map = {}

        for c in range(k_clusters):
            cluster_features = feature_clusters[feature_clusters["cluster"] == c]["feature_name"].tolist()
            cluster_features_map[c] = cluster_features

            matching_rows = ttest_results[ttest_results['feature'].isin(cluster_features)]
            N_g = len(matching_rows)
            sum_g = matching_rows['t_stat'].sum() if N_g > 0 else 0

            z_g = sum_g / math.sqrt(N_g) if N_g > 0 else 0
            p_value_bidirectional = 2 * (1 - norm.cdf(abs(z_g)))

            groups_stat_list.append({
                'group': f"g{c+1}",
                'p_value': p_value_bidirectional,
                'cluster_index': c
            })

        groups_stat = pd.DataFrame(groups_stat_list)
        _, q_values, _, _ = multipletests(groups_stat['p_value'], method='fdr_bh')
        groups_stat['q_value'] = q_values

        significant_groups = groups_stat[groups_stat['q_value'] <= FDR_ALPHA]

        if len(significant_groups) == 0:   
            print("No significant cluster (q == 0) was found in this iteration. Stopping feature filtering.")
            break

        selected_features = []
        for c in significant_groups['cluster_index']:
            selected_features.extend(cluster_features_map[c])

        selected_feature_set = set(selected_features)
        ordered_selected_features = [col for col in original_feature_order if col in selected_feature_set]

        print(f"-> Number of significant clusters selected: {len(significant_groups)}")
        print(f"-> Total number of features in the new dataset: {len(ordered_selected_features)}")

        current_data = current_data[non_feature_cols + ordered_selected_features]

        if (i + 1) in checkpoints:
            snapshots[i + 1] = current_data.copy()

    # If the loop stops early, use the last available dataset for unreachable checkpoints
    for cp in checkpoints:
        if cp not in snapshots:
            print(f"Note: Iteration {cp} was not reached; the last available dataset is being used.")
            snapshots[cp] = current_data.copy()

    return snapshots


def evaluate_rf(new_cleaned_data):
    X = new_cleaned_data.drop(["data", "label"], axis=1, errors="ignore")
    y = new_cleaned_data["label"]

    
    # Random Forest (RF) model
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    scoring = {
        'accuracy': 'accuracy',
        'f1': 'f1',
        'precision': 'precision',
        'recall': 'recall',
        'roc_auc': 'roc_auc',
        'mcc': make_scorer(matthews_corrcoef)
    }

    return cross_validate(
        model, X, y,
        cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )


def mean_std(scores):
    return f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"


# 1) Run the iterations and collect datasets at the checkpoints
snapshots = run_iterations(cleaned_data, ttest_results, MANUAL_K_CLUSTERS, MAX_ITER, CHECKPOINTS)

# 2) Evaluate and print the RF results for each dataset
summary_rows = []

for n_iter in CHECKPOINTS:
    new_cleaned_data = snapshots[n_iter]
    n_features = new_cleaned_data.drop(["data", "label"], axis=1, errors="ignore").shape[1]

    cv_results = evaluate_rf(new_cleaned_data)

    print("\n" + "=" * 55)
    print(f"Number of iterations: {n_iter} | Number of features: {n_features} | Dataset shape: {new_cleaned_data.shape}")
    print("=" * 55)
    print("10-Fold CV Results (RF):")
    print(f"Accuracy     : {mean_std(cv_results['test_accuracy'])}")
    print(f"F1 Score     : {mean_std(cv_results['test_f1'])}")
    print(f"Sensitivity  : {mean_std(cv_results['test_recall'])}")
    print(f"Precision    : {mean_std(cv_results['test_precision'])}")
    print(f"AUC (ROC)    : {mean_std(cv_results['test_roc_auc'])}")
    print(f"MCC          : {mean_std(cv_results['test_mcc'])}")

    summary_rows.append({
        "iterations": n_iter,
        "n_features": n_features,
        "Accuracy": mean_std(cv_results['test_accuracy']),
        "F1": mean_std(cv_results['test_f1']),
        "Sensitivity": mean_std(cv_results['test_recall']),
        "Precision": mean_std(cv_results['test_precision']),
        "AUC": mean_std(cv_results['test_roc_auc']),
        "MCC": mean_std(cv_results['test_mcc']),
    })

# 3) Comparison table
print("\n\n--- SUMMARY TABLE ---")
summary_df = pd.DataFrame(summary_rows).set_index("iterations")
print(summary_df.to_string())

In [ ]:
# XGBoost evaluation
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from sklearn.cluster import KMeans
import xgboost as xgb
from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.model_selection import StratifiedKFold, cross_validate

MANUAL_K_CLUSTERS = 10
FDR_ALPHA = 0.05
CHECKPOINTS = [2, 3, 4]          
MAX_ITER = max(CHECKPOINTS)


def run_iterations(cleaned_data, ttest_results, k_clusters, max_iter, checkpoints):
    """Run recursive clustering and return checkpoint snapshots."""
    current_data = cleaned_data.copy()
    original_feature_order = [col for col in cleaned_data.columns if col != 'label']
    snapshots = {}

    for i in range(max_iter):
        print(f"\n[{i+1}/{max_iter}] Iteration started...")

        non_feature_cols = [col for col in ['label'] if col in current_data.columns]
        feature_cols_df = current_data.drop(columns=non_feature_cols)
        X_features = feature_cols_df.T

        if len(X_features) < k_clusters:
            print(f"Warning: Remaining feature count ({len(X_features)}) is below k_clusters ({k_clusters}). Stopping the loop.")
            break

        print(f"-> {len(X_features)} features, partitioned into {k_clusters} clusters using KMeans...")
        kmeans = KMeans(n_clusters=k_clusters, n_init=10, random_state=42)
        cluster_labels = kmeans.fit_predict(X_features)

        feature_clusters = pd.DataFrame({
            "feature_name": X_features.index,
            "cluster": cluster_labels
        })

        groups_stat_list = []
        cluster_features_map = {}

        for c in range(k_clusters):
            cluster_features = feature_clusters[feature_clusters["cluster"] == c]["feature_name"].tolist()
            cluster_features_map[c] = cluster_features

            matching_rows = ttest_results[ttest_results['feature'].isin(cluster_features)]
            N_g = len(matching_rows)
            sum_g = matching_rows['t_stat'].sum() if N_g > 0 else 0

            z_g = sum_g / math.sqrt(N_g) if N_g > 0 else 0
            p_value_bidirectional = 2 * (1 - norm.cdf(abs(z_g)))

            groups_stat_list.append({
                'group': f"g{c+1}",
                'p_value': p_value_bidirectional,
                'cluster_index': c
            })

        groups_stat = pd.DataFrame(groups_stat_list)
        _, q_values, _, _ = multipletests(groups_stat['p_value'], method='fdr_bh')
        groups_stat['q_value'] = q_values

        significant_groups = groups_stat[groups_stat['q_value'] <= FDR_ALPHA]

        if len(significant_groups) == 0:
            print("No significant cluster met the FDR threshold in this iteration. Stopping feature filtering.")
            break

        selected_features = []
        for c in significant_groups['cluster_index']:
            selected_features.extend(cluster_features_map[c])

        selected_feature_set = set(selected_features)
        ordered_selected_features = [col for col in original_feature_order if col in selected_feature_set]

        print(f"-> Number of significant clusters selected: {len(significant_groups)}")
        print(f"-> Total number of features in the new dataset: {len(ordered_selected_features)}")

        current_data = current_data[non_feature_cols + ordered_selected_features]

        if (i + 1) in checkpoints:
            snapshots[i + 1] = current_data.copy()

    # If the loop stops early, use the last available dataset for unreachable checkpoints.
    for cp in checkpoints:
        if cp not in snapshots:
            print(f"Note: Iteration {cp} was not reached; the last available dataset is being used.")
            snapshots[cp] = current_data.copy()

    return snapshots


def evaluate_xgboost(new_cleaned_data):
    X = new_cleaned_data.drop(["data", "label"], axis=1, errors="ignore")
    y = new_cleaned_data["label"]

    # XGBoost evaluationoost modeli 
    model = xgb.XGBClassifier( 
    n_estimators=100,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=1  # Recommended for compatibility with scikit-learn.
    )
 
    scoring = {
        'accuracy': 'accuracy',
        'f1': 'f1',
        'precision': 'precision',
        'recall': 'recall',
        'roc_auc': 'roc_auc',
        'mcc': make_scorer(matthews_corrcoef)
    }

    return cross_validate(
        model, X, y,
        cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )


def mean_std(scores):
    return f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"


# 1) Run the iterations and collect checkpoint datasets.
snapshots = run_iterations(cleaned_data, ttest_results, MANUAL_K_CLUSTERS, MAX_ITER, CHECKPOINTS)

# 2) Evaluate and print SVM results for each checkpoint dataset.
summary_rows = []

for n_iter in CHECKPOINTS:
    new_cleaned_data = snapshots[n_iter]
    n_features = new_cleaned_data.drop(["data", "label"], axis=1, errors="ignore").shape[1]

    cv_results = evaluate_xgboost(new_cleaned_data)

    print("\n" + "=" * 55)
    print(f"Iteration count: {n_iter} | Feature count: {n_features} | Dataset shape: {new_cleaned_data.shape}")
    print("=" * 55)
    print("10-Fold CV Results (XGBoost):")
    print(f"Accuracy     : {mean_std(cv_results['test_accuracy'])}")
    print(f"F1 Score     : {mean_std(cv_results['test_f1'])}")
    print(f"Sensitivity  : {mean_std(cv_results['test_recall'])}")
    print(f"Precision    : {mean_std(cv_results['test_precision'])}")
    print(f"AUC (ROC)    : {mean_std(cv_results['test_roc_auc'])}")
    print(f"MCC          : {mean_std(cv_results['test_mcc'])}")

    summary_rows.append({
        "iterations": n_iter,
        "n_features": n_features,
        "Accuracy": mean_std(cv_results['test_accuracy']),
        "F1": mean_std(cv_results['test_f1']),
        "Sensitivity": mean_std(cv_results['test_recall']),
        "Precision": mean_std(cv_results['test_precision']),
        "AUC": mean_std(cv_results['test_roc_auc']),
        "MCC": mean_std(cv_results['test_mcc']),
    })

# 3) Comparison table.
print("\n\n--- SUMMARY TABLE ---")
summary_df = pd.DataFrame(summary_rows).set_index("iterations")
print(summary_df.to_string())